Para a execução, é preciso subir o arquivo do Corpus (https://github.com/roneysco/Fake.br-Corpus) zipado para a aba de arquivos do colab. Assim tendo feito, a leitura dos dados se dá no processo de "unzip" e então seguindo com os passos necessários.

In [ ]:
import nltk
import string
from nltk.tokenize import word_tokenize
import re

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')
nltk.download('punkt_tab')

#Base de Dados escolhida: Fake.Br Corpus - Universidade de São Paulo


In [ ]:
# Passos inicias para descompactar o arquivo "zippado".
import tarfile

with tarfile.open('full_texts.tar.gz', 'r:gz') as tar:
    tar.extractall('.')


In [ ]:
# Importar base de dados
import pandas as pd
import os

noticias = []

# Lendo notícias verdadeiras
for noticia in os.listdir('./full_texts/true'):
  with open(f'full_texts/true/{noticia}', 'r', encoding='utf-8') as f:
    noticias.append({'noticia':f.read(), 'categoria':'verdadeira'})

# Lendo notícias falsas
for noticia in os.listdir('./full_texts/fake'):
  with open(f'full_texts/fake/{noticia}', 'r', encoding='utf-8') as f:
    noticias.append({'noticia':f.read(), 'categoria':'falsa'})

df = pd.DataFrame(noticias)
display(df)

In [ ]:
# Tratamento para aspas 'unicode'
extras = "“”‘’"
pontuacao_completa = string.punctuation + extras

# Normalização
df['normalizacao'] = [n.lower() for n in df['noticia']]

# Remoção de números
df['normalizacao'] = [re.sub(r'\d+', '', n) for n in df['normalizacao']]

# Remoção de pontuações
df['normalizacao'] = [n.translate(str.maketrans('', '', pontuacao_completa)) for n in df['normalizacao']]

In [ ]:
# Tokenização

df['tokenizacao'] = [word_tokenize(n, language='portuguese') for n in df['normalizacao']]

In [ ]:
display(df)

In [ ]:
contracoes = {

    # de + artigos
    "do": ["de", "o"],
    "da": ["de", "a"],
    "dos": ["de", "os"],
    "das": ["de", "as"],

    # de + artigos indefinidos
    "dum": ["de", "um"],
    "duma": ["de", "uma"],
    "duns": ["de", "uns"],
    "dumas": ["de", "umas"],

    # de + demonstrativos
    "deste": ["de", "este"],
    "desta": ["de", "esta"],
    "destes": ["de", "estes"],
    "destas": ["de", "estas"],

    "desse": ["de", "esse"],
    "dessa": ["de", "essa"],
    "desses": ["de", "esses"],
    "dessas": ["de", "essas"],

    "daquele": ["de", "aquele"],
    "daquela": ["de", "aquela"],
    "daqueles": ["de", "aqueles"],
    "daquelas": ["de", "aquelas"],

    # de + pronomes neutros
    "disto": ["de", "isto"],
    "disso": ["de", "isso"],
    "daquilo": ["de", "aquilo"],


    # em + artigos
    "no": ["em", "o"],
    "na": ["em", "a"],
    "nos": ["em", "os"],
    "nas": ["em", "as"],

    # em + indefinidos
    "num": ["em", "um"],
    "numa": ["em", "uma"],
    "nuns": ["em", "uns"],
    "numas": ["em", "umas"],

    # em + demonstrativos
    "neste": ["em", "este"],
    "nesta": ["em", "esta"],
    "nestes": ["em", "estes"],
    "nestas": ["em", "estas"],

    "nesse": ["em", "esse"],
    "nessa": ["em", "essa"],
    "nesses": ["em", "esses"],
    "nessas": ["em", "essas"],

    "naquele": ["em", "aquele"],
    "naquela": ["em", "aquela"],
    "naqueles": ["em", "aqueles"],
    "naquelas": ["em", "aquelas"],

    # em + pronomes neutros
    "nisto": ["em", "isto"],
    "nisso": ["em", "isso"],
    "naquilo": ["em", "aquilo"],


    # a + artigos
    "ao": ["a", "o"],
    "aos": ["a", "os"],
    "à": ["a", "a"],
    "às": ["a", "as"],

    # a + demonstrativos
    "àquele": ["a", "aquele"],
    "àquela": ["a", "aquela"],
    "àqueles": ["a", "aqueles"],
    "àquelas": ["a", "aquelas"],

    # a + pronome neutro
    "àquilo": ["a", "aquilo"],


    # por + artigos
    "pelo": ["por", "o"],
    "pela": ["por", "a"],
    "pelos": ["por", "os"],
    "pelas": ["por", "as"]
}

tokens_expandidos = []
for tokens in df['tokenizacao']:

  expansao = []
  for token in tokens:

    if token in contracoes:
      expansao.extend(contracoes[token])

    else:
      expansao.append(token)

  tokens_expandidos.append(expansao)

df['tokens_expandidos'] = tokens_expandidos

In [ ]:
display(df['tokens_expandidos'])

In [ ]:
# Remoção de Stopwords
from nltk.corpus import stopwords
stop_words = set(stopwords.words('portuguese'))

df['no_stopwords'] = [[word for word in tokens if word not in stop_words] for tokens in df['tokens_expandidos']]

In [ ]:
display(df['no_stopwords'][0])

In [ ]:
# Stemming / Lematização / Radicalização
from nltk.stem import RSLPStemmer
nltk.download('rslp')

stemmer = RSLPStemmer()
df['stemming'] = [[stemmer.stem(t) for t in tokens] for tokens in df['no_stopwords']]

In [ ]:
display(df['stemming'])

#Etapa 3 - Análise dos Textos

In [ ]:
# Número total de tokens - 6a
import plotly.express as px

# Usei uma list para armazenar o número total de tokens de cada coluna
df['total_tokens'] = [len(tokens) for tokens in df['no_stopwords']]

# plotei com plotly.
fig = px.histogram(df, x='total_tokens', text_auto=True, marginal="box", title="Distribuição de Tokens")

fig.update_layout(
    xaxis_title="Número de Tokens",
    yaxis_title="Quantidade de Documentos",
    bargap=0.05
)
fig.show()

6b - Com base na visualização do histograma, vê-se que a maioria dos textos são curtos. Observando o boxplot superior, notamos que a caixa (intervalo interquartil) é bastante estreita e está bem pra esquerda. Ou seja, a metade de todas as notícias fica abaixo de um limite de tokens muito pequeno.

In [ ]:
# 6c - Separar de acordo com as classes do conjunto de dados e responder se para cada classe, a distribuição é curta ou longa.

fig = px.histogram(
    df,
    x='total_tokens',
    color='categoria',
    marginal='box',
    barmode='overlay',
    title='Comparação de tamanhos')

fig.update_layout(
    xaxis_title="Número de Tokens",
    yaxis_title="Quantidade de Documentos",
    bargap=0.05
)
fig.show()

6c - Pode-se concluir também que, dado o histograma acima, vê-se que ambos são curtos. Até mesmo olhando para o boxplot, ambas as categorias de noticias estão estreita à esquerda.

In [ ]:
# 7a - identificando termos mais comuns
from collections import Counter

processadas = []

for n in df['no_stopwords']:
  processadas.extend(n)

contagem_processadas = Counter(processadas)

display(contagem_processadas.most_common())

In [ ]:
# identificando os 10 mais comuns
top_10 = contagem_processadas.most_common(10)

df_top10 = pd.DataFrame(top_10, columns=['palavra', 'frequencia'])
fig = px.histogram(
    df_top10,
    x='palavra',
    y='frequencia',
    text_auto=True,
    title='Palavras mais comuns'
)

fig.show()

top10_antes = df['tokenizacao'].explode().value_counts().head(10).reset_index()
top10_antes.columns = ['palavra', 'frequencia']

fig_antes = px.bar(
    top10_antes,
    x='palavra',
    y='frequencia',
    text_auto=True,
    title='Top 10 Palavras Mais Frequentes antes do processamento'
)
fig_antes.show()

As 10 palavras mais presentes no corpus após todo o pré-processamento estão listadas no primeiro gráfico.

7c - O gráfico antes do pré-processamento mostra que as palavras mais comuns eram de fato as stopwors que durante o pre-processamento são removidas, fazendo assim com que as palaras mais comuns e que têm relevância sejam as 10 do primeiro histograma.

7d - Lula, presidente, governo e federal estando essas entre as palavras mais citadas, ressaltam e mostram que o tema predominante é o âmbito da política (É o foco das notícias do Corpus)

7e - Com base na análise de frequência, os temas que aparecem com maior destaque no corpus estão concentrados em Política Nacional.

Isso é evidenciado pela alta recorrência de termos como "Lula", "Presidente" e "Governo". Analisando esse vocabulário, podemos inferir que os principais problemas retratados nos textos envolvem as falas e ações do governo Lula neste tempo.

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

noticias = " ".join(processadas)

wc = WordCloud(background_color='white', width=800, height=400, repeat=True)
wc.generate_from_frequencies(contagem_processadas)

fig = px.imshow(wc)
fig.show()


In [ ]:
df_verdadeiras = df[df['categoria'] == 'verdadeira']
df_falsas = df[df['categoria'] == 'falsa']

freq_verdadeiras = df_verdadeiras['no_stopwords'].dropna().explode().value_counts().head(10).reset_index()
freq_verdadeiras.columns = ['Palavra', 'Frequência']

freq_falsas = df_falsas['no_stopwords'].dropna().explode().value_counts().head(10).reset_index()
freq_falsas.columns = ['Palavra', 'Frequência']

fig_v = px.bar(
    freq_verdadeiras, x='Palavra', y='Frequência', text_auto=True,
    title='Top 10 Palavras - Notícias Verdadeiras',
    color_discrete_sequence=['#2ca02c'] #Verde
)
fig_v.show()

fig_f = px.bar(
    freq_falsas, x='Palavra', y='Frequência', text_auto=True,
    title='Top 10 Palavras - Notícias Falsas',
    color_discrete_sequence=['#d62728'] #Vermelho
)
fig_f.show()

8a - O vocabulário aparenta ser bem semelhante, no entanto uma diferença interessante é que, ao se referir ao Lula, as notícias verdadeiras trazem também o adjetivo "presidente" acompanhado e/ou apenas o termo presidente, enquanto as falsas, é notável a ausência desse adjetivo pela sua baixa frequência em relação às verdadeiras.

In [ ]:
from wordcloud import WordCloud
import plotly.graph_objects as go
from plotly.subplots import make_subplots

dict_v = df_verdadeiras['no_stopwords'].dropna().explode().value_counts().to_dict()
dict_f = df_falsas['no_stopwords'].dropna().explode().value_counts().to_dict()

nuvem_v = WordCloud(width=800, height=500, background_color='white', colormap='Greens', max_words=100).generate_from_frequencies(dict_v)
nuvem_f = WordCloud(width=800, height=500, background_color='white', colormap='Reds', max_words=100).generate_from_frequencies(dict_f)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Notícias Verdadeiras", "Notícias Falsas"),
    horizontal_spacing=0.05
)

fig.add_trace(go.Image(z=nuvem_v.to_array()), row=1, col=1)
fig.add_trace(go.Image(z=nuvem_f.to_array()), row=1, col=2)

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

fig.update_layout(
    title_text="Comparação de Vocabulário por Classe",
    title_x=0.5,
    height=500,
    margin=dict(l=10, r=10, t=80, b=10) # Reduz as margens para as imagens ficarem maiores
)

fig.show()

8b - Uma visualização dos termos mais importantes de cada classe com as nuvens de palavras lado a lado para uma melhor percepção.